# DraftScope 10-year college model and active-roster benchmark audit

## tl;dr

This notebook independently audits two deliberately separate populations:

1. the complete 2016–2025 FBS roster checkpoints used only for `P(drafted in the immediately following NFL draft | FBS roster player at the checkpoint)`; and
2. the 2017–2026 drafted-player benchmark joined to the current 2026 nflverse roster snapshot. Displayed averages and elite/top-64 comparisons use current-roster survivors only.

The notebook hard-fails on a stale or incomplete ten-year artifact, a training CSV hash/byte mismatch, nonconsecutive partitions, per-row probability-contract drift, failed strict gates, unresolved identity conflicts, future-year model training, or current-roster/NFL-outcome fields entering the probability feature allow-list. It separately audits alias collapse, explicit college missingness, fixed three-year NFL contributor outcomes, expanding-window validation, optional CFBD success features, and optional immutable tracking artifacts. Optional layers report `UNAVAILABLE` when their source artifacts have not been produced yet. Headline values and tables are generated from current artifacts rather than hard-coded.

In [1]:
from collections import Counter, defaultdict
from pathlib import Path
from statistics import fmean
import csv
import hashlib
import json
import sys

EXPECTED_COLLEGE_SEASONS = tuple(range(2016, 2026))
EXPECTED_DRAFT_YEARS = tuple(range(2017, 2027))
EXPECTED_LOOKBACK_YEARS = 10
EXPECTED_SOURCE_AUDITS = {
    'sportsdataverse': {
        'reviewed_omissions': 12,
        'artifact_rows': 154_279,
        'drafted_rows': 2_404,
        'positive_alias_rows': 1,
        'negative_alias_rows': 45,
        'within_id_duplicate_rows': 140,
    },
    'cfbd': {
        'reviewed_omissions': 7,
        'artifact_rows': 153_222,
        'drafted_rows': 2_416,
        'positive_alias_rows': 33,
        'negative_alias_rows': 470,
        'within_id_duplicate_rows': 221,
    },
}
ACTIVE_ROSTER_SEASON = 2026
EXPECTED_ACTIVE_STATUSES = frozenset({'ACT', 'RES', 'E14', 'INA', 'PUP', 'SUS', 'EXE', 'DEV'})
RUN_MODEL_VALIDATION = True
VALIDATION_POSITIONS = None

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'draftscope').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the DraftScope checkout.')

def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def missing(value):
    return value is None or str(value).strip().lower() in {'', 'na', 'n/a', 'none', 'null', 'nan'}

def truthy(value):
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}

def percentage(numerator, denominator, digits=1):
    return f'{numerator / denominator:.{digits}%}' if denominator else 'n/a'

def print_table(table_rows, columns=None):
    if not table_rows:
        print('(no rows)')
        return
    columns = list(columns or table_rows[0].keys())
    rendered = [[str(row.get(column, '')) for column in columns] for row in table_rows]
    widths = [max(len(column), *(len(row[index]) for row in rendered)) for index, column in enumerate(columns)]
    print(' | '.join(column.ljust(widths[index]) for index, column in enumerate(columns)))
    print('-+-'.join('-' * width for width in widths))
    for row in rendered:
        print(' | '.join(value.ljust(widths[index]) for index, value in enumerate(row)))

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def require(condition, message):
    if not condition:
        raise AssertionError(message)

REPO_ROOT = find_repo_root(Path.cwd())
HISTORY_PATH = REPO_ROOT / 'data' / 'model_history' / 'college_history.csv'
METADATA_PATH = HISTORY_PATH.with_suffix('.metadata.json')
CACHE_DIR = REPO_ROOT / '.draftscope-cache'
COMBINE_PATH = CACHE_DIR / 'combine.csv'
DRAFT_PATH = CACHE_DIR / 'draft_picks.csv'
ROSTER_PATH = CACHE_DIR / f'roster_{ACTIVE_ROSTER_SEASON}.csv'
TRACKING_OUTPUT_DIR = REPO_ROOT / 'data'

if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

from draftscope.college_features import CFBD_GENERATED_PRODUCTION_FIELDS, derive_player_success
from draftscope.data_sources import ACTIVE_NFL_ROSTER_STATUSES, load_nflverse_history
from draftscope.historical_college import audit_historical_college_training_data
from draftscope.model import COLLEGE_AVAILABILITY_FEATURES, COLLEGE_CONTEXT_FEATURES, CollegeDraftProbabilityModel, ModelError, college_candidate_features
from draftscope.records import load_records, normalize_name, parse_bool, parse_number
from draftscope.schema import POSITION_GROUPS, normalize_position
from draftscope.tracking import TrackingStore

for required_path in (HISTORY_PATH, METADATA_PATH, COMBINE_PATH, DRAFT_PATH, ROSTER_PATH):
    require(required_path.is_file(), f'Missing required artifact: {required_path}')

history_rows = read_csv(HISTORY_PATH)
metadata = read_json(METADATA_PATH)
COLLEGE_PROVIDER = str(metadata.get('college_data_provider') or '').strip().lower()
require(COLLEGE_PROVIDER in EXPECTED_SOURCE_AUDITS, f'No reviewed audit baseline for college provider: {COLLEGE_PROVIDER!r}')
source_audit = EXPECTED_SOURCE_AUDITS[COLLEGE_PROVIDER]
EXPECTED_REVIEWED_ROSTER_OMISSIONS = source_audit['reviewed_omissions']
EXPECTED_ARTIFACT_ROWS = source_audit['artifact_rows']
EXPECTED_DRAFTED_ROWS = source_audit['drafted_rows']
EXPECTED_POSITIVE_ALIAS_ROWS = source_audit['positive_alias_rows']
EXPECTED_NEGATIVE_ALIAS_ROWS = source_audit['negative_alias_rows']
EXPECTED_WITHIN_ID_DUPLICATE_ROWS = source_audit['within_id_duplicate_rows']
require(tuple(metadata.get('college_seasons', ())) == EXPECTED_COLLEGE_SEASONS, 'Refusing stale/non-10-year college history sidecar.')
require(tuple(metadata.get('draft_years', ())) == EXPECTED_DRAFT_YEARS, 'College-to-draft year contract is not 2016–2025 -> 2017–2026.')
require(metadata.get('quality_gate_status') == 'pass', 'Strict historical quality gate is not pass.')
require(metadata.get('contract_audit', {}).get('status') == 'pass', 'Historical contract audit is not pass.')
require(not metadata.get('source_failures'), f'Historical source failures remain: {metadata.get("source_failures")}')
require(int(metadata.get('reviewed_roster_source_omission_count') or 0) == EXPECTED_REVIEWED_ROSTER_OMISSIONS, f'Reviewed roster omission count drifted from the {COLLEGE_PROVIDER} audit baseline.')
training_artifact = dict(metadata.get('training_artifact') or {})
actual_history_bytes = HISTORY_PATH.stat().st_size
actual_history_sha256 = file_sha256(HISTORY_PATH)
require(int(training_artifact.get('bytes') or -1) == actual_history_bytes, 'Training CSV byte count does not match its sidecar.')
require(training_artifact.get('sha256') == actual_history_sha256, 'Training CSV SHA-256 does not match its sidecar.')
require(frozenset(ACTIVE_NFL_ROSTER_STATUSES) == EXPECTED_ACTIVE_STATUSES, 'Active roster status contract changed unexpectedly.')

benchmark_history, benchmark_metadata = load_nflverse_history(
    CACHE_DIR,
    lookback_years=EXPECTED_LOOKBACK_YEARS,
    end_year=EXPECTED_DRAFT_YEARS[-1],
    active_roster_season=ACTIVE_ROSTER_SEASON,
    nfl_completed_season=ACTIVE_ROSTER_SEASON - 1,
    refresh=False,
)
require(int(benchmark_metadata.get('window_start') or 0) == EXPECTED_DRAFT_YEARS[0], 'Benchmark window start is not 2017.')
require(int(benchmark_metadata.get('window_end') or 0) == EXPECTED_DRAFT_YEARS[-1], 'Benchmark window end is not 2026.')
require(int(benchmark_metadata.get('lookback_years') or 0) == EXPECTED_LOOKBACK_YEARS, 'Benchmark lookback is not ten drafts.')
require(frozenset(benchmark_metadata.get('active_roster_included_statuses') or ()) == EXPECTED_ACTIVE_STATUSES, 'Benchmark active-status population differs from the contract.')

combine_raw = read_csv(COMBINE_PATH)
draft_raw = read_csv(DRAFT_PATH)
roster_raw = read_csv(ROSTER_PATH)
drafted_rows = sum(truthy(row.get('drafted')) for row in history_rows)
expected_picks = int(metadata.get('expected_fbs_draft_picks') or 0)
matched_picks = int(metadata.get('matched_expected_fbs_draft_picks') or 0)

print(f'College artifact: {HISTORY_PATH.relative_to(REPO_ROOT)}')
print(f'Built at: {metadata.get("built_at", "unknown")}')
print(f'College provider: {COLLEGE_PROVIDER} ({metadata.get("college_source_name", "unknown source")})')
print(f'College seasons: {EXPECTED_COLLEGE_SEASONS[0]}–{EXPECTED_COLLEGE_SEASONS[-1]} ({len(EXPECTED_COLLEGE_SEASONS)} partitions)')
print(f'Population rows: {len(history_rows):,}; drafted outcome rows: {drafted_rows:,}')
print(f'Strict linkage: {matched_picks:,}/{expected_picks:,} expected FBS picks; reviewed source omissions: {EXPECTED_REVIEWED_ROSTER_OMISSIONS}')
print(f'Active benchmark: {benchmark_metadata.get("benchmark_rows", 0):,}/{benchmark_metadata.get("active_drafted_window_rows", 0):,} active drafted players with a combine/pro-day measurement')
print(f'Active roster snapshot: {ACTIVE_ROSTER_SEASON} Week {benchmark_metadata.get("active_roster_snapshot_week")}')

College artifact: data/model_history/college_history.csv
Built at: 2026-08-27T10:04:39-04:00
College provider: sportsdataverse (SportsDataverse ESPN college football releases)
College seasons: 2016–2025 (10 partitions)
Population rows: 154,279; drafted outcome rows: 2,404
Strict linkage: 2,404/2,404 expected FBS picks; reviewed source omissions: 12
Active benchmark: 1,472/1,632 active drafted players with a combine/pro-day measurement
Active roster snapshot: 2026 Week 1


## Context & Methods

The college model artifact has one row per unique FBS roster player per season. At the preseason Week 0 checkpoint, season `S` roster measurements are paired only with completed season `S-1` production, then labeled by draft `S+1`. Returning and ineligible players remain valid negatives in this unconditional risk set.

The benchmark is separate. It starts with nflverse Combine/pro-day and draft records for 2017–2026, then joins drafted players to the 2026 roster snapshot using stable identifiers and an ambiguity-rejecting fallback. Displayed “active average” and “elite active” values are conditioned on current roster survival and metric availability. College-production benchmarks use the same active drafted registry joined to drafted college outcome rows by nflverse PFR or CFB identifier.

College-stage incomplete numeric values use training-fold medians. Implicit missing-value indicators are disabled; `has_recorded_stats` is the sole explicit availability feature. The grouped model audit surfaces material source-era coverage shifts rather than learning missingness as a player-skill signal. Provider-specific known shifts are asserted only for the provider where they were established.

### Key assumptions and limitations

- nflverse draft picks are outcome truth; the selected college provider supplies an audited identity crosswalk only.
- Exact, provider-specific reviewed draft picks absent from that provider’s FBS roster snapshots are excluded from the source-defined roster denominator and audited separately. New omissions fail the strict build.
- The training CSV byte count and SHA-256 must match the integrity record in its JSON sidecar before any analysis runs.
- Per-row probability kind and condition must exactly match the sidecar contract.
- 2021 testing was decentralized pro-day testing, not the normal in-person Combine protocol.
- Current-roster averages have survivorship and recency bias. Recent classes have had less time to attrit, while injuries, retirement, releases, and missing Combine tests selectively remove older observations.
- Active benchmarks are descriptive comparisons only. Active status, current team, and benchmark membership are post-outcome fields and must never enter the college probability model.

In [2]:
print('Checkpoint and population contract')
for field in ('model_stage', 'probability_kind', 'probability_condition', 'population', 'checkpoint', 'as_of_week', 'row_population'):
    print(f'- {field}: {metadata.get(field)}')
print(f'- college seasons: {metadata.get("college_seasons")}')
print(f'- draft years: {metadata.get("draft_years")}')
print(f'- feature cutoff: {metadata.get("leakage_controls", {}).get("feature_source_cutoff")}')
print(f'- benchmark population: {benchmark_metadata.get("benchmark_population")}')
print(f'- active roster retrieved: {benchmark_metadata.get("active_roster_source_as_of")} ({benchmark_metadata.get("active_roster_source_as_of_basis")})')
print(f'- included active statuses: {benchmark_metadata.get("active_roster_included_statuses")}')
print(f'- training artifact integrity: {actual_history_bytes:,} bytes; SHA-256 {actual_history_sha256} (sidecar match PASS)')

source_rows = []
raw_sources = ((COMBINE_PATH, len(combine_raw)), (DRAFT_PATH, len(draft_raw)), (ROSTER_PATH, len(roster_raw)))
for path, rows in raw_sources:
    source_sidecar_path = path.with_suffix(path.suffix + '.source.json')
    require(source_sidecar_path.is_file(), f'Missing source sidecar: {source_sidecar_path}')
    source_sidecar = read_json(source_sidecar_path)
    actual_hash = file_sha256(path)
    require(source_sidecar.get('sha256') == actual_hash, f'SHA-256 mismatch for {path.name}')
    source_rows.append({
        'artifact': path.name,
        'rows': f'{rows:,}',
        'retrieved': source_sidecar.get('downloaded_at', 'unknown'),
        'sha256': actual_hash[:16],
        'hash': 'PASS',
    })
require(benchmark_metadata.get('active_roster_source_sha256') == file_sha256(ROSTER_PATH), 'Benchmark metadata roster hash does not match the local source.')
source_rows.insert(0, {
    'artifact': str(HISTORY_PATH.relative_to(REPO_ROOT)),
    'rows': f'{len(history_rows):,}',
    'retrieved': metadata.get('built_at', 'unknown'),
    'sha256': actual_history_sha256[:16],
    'hash': 'PASS',
})
print('\nLocal source artifact audit')
print_table(source_rows)

college_source_rows = []
for artifact in metadata.get('source_artifacts', []):
    params = artifact.get('params') or {}
    college_source_rows.append({
        'year': params.get('year', ''),
        'source': artifact.get('endpoint') or artifact.get('source', ''),
        'rows': artifact.get('rows', ''),
        'sha256': str(artifact.get('sha256') or '')[:16],
        'cache': artifact.get('cache_hit', ''),
    })
require(college_source_rows and all(row['sha256'] for row in college_source_rows), 'A college source artifact lacks a recorded SHA-256.')
print('\nCollege source lineage (metadata-recorded payload hashes)')
print_table(college_source_rows)

Checkpoint and population contract
- model_stage: college_precombine
- probability_kind: unconditional_next_draft
- probability_condition: P(drafted in immediately following NFL draft | FBS roster player at checkpoint)
- population: All unique source-listed FBS roster players at the preseason checkpoint, with prior completed-season production where available
- checkpoint: preseason_prior_year
- as_of_week: 0
- row_population: FBS roster preseason; prior-season stats
- college seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
- draft years: [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
- feature cutoff: current roster plus prior completed season statistics
- benchmark population: active on nflverse 2026 roster; drafted 2017-2026; combine measurement available
- active roster retrieved: 2026-08-27T09:48:40-04:00 (local artifact download timestamp)
- included active statuses: ['ACT', 'DEV', 'E14', 'EXE', 'INA', 'PUP', 'RES', 'SUS']
- training artifact in

## College model data

The next cells verify raw-file grain, ten consecutive partitions, outcome linkage, reviewed omissions, alias collapsing, and position/metric coverage. Context target encodings are excluded from source coverage because they are derived inside each training fold.

In [3]:
row_ids = [row.get('row_id', '') for row in history_rows]
player_season_keys = [(row.get('player_id', ''), row.get('season', '')) for row in history_rows]
required_fields = ('row_id', 'player_id', 'name', 'season', 'draft_year', 'population', 'probability_kind', 'probability_condition', 'drafted', 'outcome_label_known')
observed_seasons = tuple(sorted({int(row['season']) for row in history_rows}))
observed_draft_years = tuple(sorted({int(row['draft_year']) for row in history_rows}))
header_fields = set(history_rows[0]) if history_rows else set()
normalized_rows = load_records(HISTORY_PATH)
normalized_contract = audit_historical_college_training_data(normalized_rows, expected_seasons=EXPECTED_COLLEGE_SEASONS)
parsed_boolean_fields = ('drafted', 'outcome_label_known', 'has_recorded_stats')

raw_checks = [
    ('CSV row count equals metadata', len(history_rows) == int(metadata.get('rows') or -1), f'{len(history_rows):,}'),
    ('Drafted row count equals metadata', drafted_rows == int(metadata.get('drafted_rows') or -1), f'{drafted_rows:,}'),
    ('Corrected artifact row count', len(history_rows) == EXPECTED_ARTIFACT_ROWS, f'{len(history_rows):,}'),
    ('Corrected artifact positive count', drafted_rows == EXPECTED_DRAFTED_ROWS, f'{drafted_rows:,}'),
    ('Exactly ten consecutive college partitions', observed_seasons == EXPECTED_COLLEGE_SEASONS, observed_seasons),
    ('Immediate draft partitions are 2017–2026', observed_draft_years == EXPECTED_DRAFT_YEARS, observed_draft_years),
    ('Duplicate row_id values', len(row_ids) == len(set(row_ids)), len(row_ids) - len(set(row_ids))),
    ('Duplicate player-season keys', len(player_season_keys) == len(set(player_season_keys)), len(player_season_keys) - len(set(player_season_keys))),
    ('Required fields populated', not any(any(missing(row.get(field)) for field in required_fields) for row in history_rows), ''),
    ('CSV schema matches sidecar', header_fields == set(metadata.get('schema_fields', [])), sorted(header_fields ^ set(metadata.get('schema_fields', [])))),
    ('Normalized parser preserves row count', len(normalized_rows) == len(history_rows), f'{len(normalized_rows):,}'),
    ('Parsed labels/availability are real booleans', all(isinstance(row.get(field), bool) for row in normalized_rows for field in parsed_boolean_fields), parsed_boolean_fields),
    ('Reloaded CSV independently passes public audit', normalized_contract.get('status') == 'pass', normalized_contract.get('failures', [])),
    ('All outcome labels known', all(truthy(row.get('outcome_label_known')) for row in history_rows), ''),
    ('Per-row probability kind matches sidecar', all(row.get('probability_kind') == metadata.get('probability_kind') for row in history_rows), metadata.get('probability_kind')),
    ('Per-row probability condition matches sidecar', all(row.get('probability_condition') == metadata.get('probability_condition') for row in history_rows), metadata.get('probability_condition')),
    ('Probability-contract audit has zero invalid rows', int(metadata.get('contract_audit', {}).get('invalid_probability_contract_rows') or 0) == 0, metadata.get('contract_audit', {}).get('invalid_probability_contract_rows')),
    ('Positive linkage coverage is 100%', matched_picks == expected_picks and float(metadata.get('positive_match_coverage') or 0.0) == 1.0, f'{matched_picks}/{expected_picks}'),
    ('Reviewed roster omissions match source audit', int(metadata.get('reviewed_roster_source_omission_count') or 0) == EXPECTED_REVIEWED_ROSTER_OMISSIONS, metadata.get('reviewed_roster_source_omission_count')),
]
print_table([{'check': name, 'status': 'PASS' if passed else 'FAIL', 'detail': detail} for name, passed, detail in raw_checks])
for name, passed, detail in raw_checks:
    require(passed, f'{name} failed: {detail}')

check                                            | status | detail                                                                         
-------------------------------------------------+--------+--------------------------------------------------------------------------------
CSV row count equals metadata                    | PASS   | 154,279                                                                        
Drafted row count equals metadata                | PASS   | 2,404                                                                          
Corrected artifact row count                     | PASS   | 154,279                                                                        
Corrected artifact positive count                | PASS   | 2,404                                                                          
Exactly ten consecutive college partitions       | PASS   | (2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025)                   
Immediate draft part

In [4]:
season_metadata = {int(item['college_season']): item for item in metadata.get('per_season', [])}
require(tuple(sorted(season_metadata)) == EXPECTED_COLLEGE_SEASONS, 'Per-season metadata partitions do not match the CSV.')
rows_by_season = defaultdict(list)
for row in history_rows:
    rows_by_season[int(row['season'])].append(row)

partition_rows = []
for season in EXPECTED_COLLEGE_SEASONS:
    cohort = rows_by_season[season]
    audit = season_metadata[season]
    positives = sum(truthy(row.get('drafted')) for row in cohort)
    partition_rows.append({
        'season': season,
        'draft': season + 1,
        'rows': f'{len(cohort):,}',
        'drafted': positives,
        'base rate': percentage(positives, len(cohort)),
        'any stats': percentage(sum(truthy(row.get('has_recorded_stats')) for row in cohort), len(cohort)),
        'height': percentage(sum(not missing(row.get('height_in')) for row in cohort), len(cohort)),
        'weight': percentage(sum(not missing(row.get('weight_lb')) for row in cohort), len(cohort)),
        'class': percentage(sum(not missing(row.get('class_year')) for row in cohort), len(cohort)),
        'pick match': percentage(int(audit.get('matched_expected_fbs_draft_picks') or 0), int(audit.get('expected_fbs_draft_picks') or 0)),
        'omissions': int(audit.get('reviewed_roster_source_omission_count') or 0),
    })
print_table(partition_rows)

season | draft | rows   | drafted | base rate | any stats | height | weight | class | pick match | omissions
-------+-------+--------+---------+-----------+-----------+--------+--------+-------+------------+----------
2016   | 2017  | 14,864 | 229     | 1.5%      | 17.8%     | 98.7%  | 98.6%  | 98.7% | 100.0%     | 1        
2017   | 2018  | 14,882 | 233     | 1.6%      | 22.4%     | 99.1%  | 99.0%  | 99.0% | 100.0%     | 0        
2018   | 2019  | 15,037 | 237     | 1.6%      | 22.3%     | 99.3%  | 99.3%  | 99.3% | 100.0%     | 0        
2019   | 2020  | 15,168 | 244     | 1.6%      | 21.3%     | 99.3%  | 99.5%  | 99.4% | 100.0%     | 1        
2020   | 2021  | 14,982 | 241     | 1.6%      | 20.9%     | 99.5%  | 99.6%  | 99.5% | 100.0%     | 8        
2021   | 2022  | 15,887 | 237     | 1.5%      | 21.9%     | 99.4%  | 99.5%  | 99.6% | 100.0%     | 0        
2022   | 2023  | 15,681 | 243     | 1.5%      | 29.3%     | 99.7%  | 99.8%  | 99.8% | 100.0%     | 1        
2023   | 2024  | 15

In [5]:
def public_source_observed(row, feature):
    if feature == 'bmi':
        return not missing(row.get('height_in')) and not missing(row.get('weight_lb'))
    if feature == 'class_year_numeric':
        return not missing(row.get('class_year'))
    return not missing(row.get(feature))

college_coverage_rows = []
for position in POSITION_GROUPS:
    cohort = [row for row in history_rows if row.get('position') == position]
    if not cohort:
        continue
    source_features = [feature for feature in college_candidate_features(position) if feature not in COLLEGE_CONTEXT_FEATURES]
    observed_inputs = sum(public_source_observed(row, feature) for row in cohort for feature in source_features)
    college_coverage_rows.append({
        'pos': position,
        'model pool': 'OT+IOL' if position in {'OT', 'IOL'} else position,
        'rows': f'{len(cohort):,}',
        'drafted': sum(truthy(row.get('drafted')) for row in cohort),
        'height': percentage(sum(not missing(row.get('height_in')) for row in cohort), len(cohort)),
        'weight': percentage(sum(not missing(row.get('weight_lb')) for row in cohort), len(cohort)),
        'class': percentage(sum(not missing(row.get('class_year')) for row in cohort), len(cohort)),
        'age': percentage(sum(not missing(row.get('age_at_draft')) for row in cohort), len(cohort)),
        'any stats': percentage(sum(truthy(row.get('has_recorded_stats')) for row in cohort), len(cohort)),
        'source inputs': percentage(observed_inputs, len(cohort) * len(source_features)),
    })
print_table(college_coverage_rows)
unsupported_rows = sum(row.get('position') not in POSITION_GROUPS for row in history_rows)
print(f'\nUnsupported/unnormalized position rows retained for audit: {unsupported_rows:,} ({percentage(unsupported_rows, len(history_rows))})')

pos  | model pool | rows   | drafted | height | weight | class  | age   | any stats | source inputs
-----+------------+--------+---------+--------+--------+--------+-------+-----------+--------------
QB   | QB         | 7,476  | 106     | 99.8%  | 99.9%  | 99.9%  | 11.1% | 37.1%     | 28.0%        
RB   | RB         | 11,386 | 197     | 99.7%  | 99.8%  | 99.9%  | 12.9% | 45.8%     | 31.0%        
WR   | WR         | 20,878 | 313     | 99.8%  | 99.8%  | 99.8%  | 14.0% | 38.0%     | 29.8%        
TE   | TE         | 8,998  | 143     | 99.8%  | 99.9%  | 99.9%  | 14.0% | 32.5%     | 28.8%        
OT   | OT+IOL     | 917    | 24      | 100.0% | 100.0% | 99.9%  | 18.5% | 2.3%      | 34.6%        
IOL  | OT+IOL     | 24,009 | 373     | 99.8%  | 99.9%  | 100.0% | 13.0% | 1.8%      | 34.2%        
EDGE | EDGE       | 5,274  | 125     | 99.8%  | 99.8%  | 99.8%  | 15.8% | 21.6%     | 30.8%        
IDL  | IDL        | 17,183 | 300     | 99.8%  | 99.8%  | 99.9%  | 14.1% | 21.1%     | 30.7%        


### Optional checkpoint success-feature audit

Some providers expose checkpoint-bounded passing/rushing success metrics. When the six generated columns contain observations, this cell reports college-season coverage, verifies the shared historical/live transformer contract, and reconciles success rates to successes divided by opportunities. Providers without populated success metrics report `UNAVAILABLE` without weakening the strict base-artifact checks.

In [6]:
SUCCESS_FEATURES = (
    'prod_passing_success_plays', 'prod_passing_successes', 'prod_passing_success_rate',
    'prod_rushing_success_plays', 'prod_rushing_successes', 'prod_rushing_success_rate',
)
success_fields_present = tuple(field for field in SUCCESS_FEATURES if field in header_fields)
success_schema_complete = set(SUCCESS_FEATURES) <= header_fields
success_observed_values = sum(not missing(row.get(field)) for row in history_rows for field in SUCCESS_FEATURES)
success_feature_audit_available = success_schema_complete and success_observed_values > 0
success_coverage_summary = {}
success_parity_violations = []
if not success_feature_audit_available:
    missing_success_fields = sorted(set(SUCCESS_FEATURES) - header_fields)
    if missing_success_fields:
        print(f'UNAVAILABLE: college history contains {len(success_fields_present)}/{len(SUCCESS_FEATURES)} checkpoint success fields.')
        print('Missing schema fields:', ', '.join(missing_success_fields))
    else:
        print(f'UNAVAILABLE for {COLLEGE_PROVIDER}: the six checkpoint success fields contain no observed values.')
else:
    require(set(SUCCESS_FEATURES) <= set(CFBD_GENERATED_PRODUCTION_FIELDS), 'Success fields are not owned by the shared transformer.')
    probe = derive_player_success({
        'passing': {'plays': 10, 'successes': 6, 'successRate': 0.6},
        'rushing': {'plays': 8, 'successes': 4, 'successRate': 0.5},
    })
    require(set(probe) == set(SUCCESS_FEATURES), f'Shared success transformer output drifted: {sorted(probe)}')
    for row in history_rows:
        for prefix in ('passing', 'rushing'):
            plays = parse_number(row.get(f'prod_{prefix}_success_plays'))
            successes = parse_number(row.get(f'prod_{prefix}_successes'))
            rate = parse_number(row.get(f'prod_{prefix}_success_rate'))
            if any(value is not None and value < 0 for value in (plays, successes, rate)):
                success_parity_violations.append((row.get('row_id'), prefix, 'negative value'))
                continue
            if rate is not None and rate > 1:
                success_parity_violations.append((row.get('row_id'), prefix, 'rate outside [0,1]'))
            if plays is not None and successes is not None and successes > plays:
                success_parity_violations.append((row.get('row_id'), prefix, 'successes exceed plays'))
            if plays is not None and plays > 0 and successes is not None and rate is not None:
                delta = abs(rate - successes / plays)
                if delta > 0.005:
                    success_parity_violations.append((row.get('row_id'), prefix, f'rate delta={delta:.6f}'))
    require(not success_parity_violations, f'Checkpoint success parity violations: {success_parity_violations[:5]}')
    success_coverage_rows = []
    for season in EXPECTED_COLLEGE_SEASONS:
        cohort = [row for row in history_rows if int(row.get('season') or 0) == season]
        item = {'season': season, 'rows': f'{len(cohort):,}'}
        for prefix in ('passing', 'rushing'):
            item[f'{prefix} plays'] = percentage(sum(not missing(row.get(f'prod_{prefix}_success_plays')) for row in cohort), len(cohort))
            item[f'{prefix} rate'] = percentage(sum(not missing(row.get(f'prod_{prefix}_success_rate')) for row in cohort), len(cohort))
        success_coverage_rows.append(item)
    for field in SUCCESS_FEATURES:
        observed = sum(not missing(row.get(field)) for row in history_rows)
        success_coverage_summary[field] = {'rows': observed, 'coverage': observed / len(history_rows) if history_rows else None}
    print('Shared historical/live transformer contract: PASS')
    print(f'Rate/opportunity parity: PASS ({len(success_parity_violations)} violations)')
    print_table(success_coverage_rows)
    print()
    print('Overall feature coverage')
    print_table([{'feature': field, 'rows': f"{detail['rows']:,}", 'coverage': f"{detail['coverage']:.1%}"} for field, detail in success_coverage_summary.items()])

UNAVAILABLE for sportsdataverse: the six checkpoint success fields contain no observed values.


### Outcome linkage, reviewed omissions, and alias collapse

Reviewed roster-source omissions are not silently manufactured as negatives or positive rows. They are exact audited exceptions to the expected-link denominator. Alias resolution must collapse duplicate roster identities before the final player-season grain is emitted.

In [7]:
method_totals = Counter()
identity_conflicts = []
unmatched_expected = []
duplicate_outcomes = []
reviewed_omissions = []
resolved_aliases = []
alias_rows_dropped = 0
negative_alias_rows_dropped = 0
negative_alias_groups_collapsed = 0
negative_alias_candidate_groups = 0
negative_alias_unique_stat_groups = 0
negative_alias_multiple_stat_groups = 0
negative_alias_weak_identity_pairs = 0
negative_alias_dropped_keys = []
negative_alias_selected_keys = []
within_id_duplicate_rows_dropped = 0
alias_conflicts = []
linkage_rows = []
final_player_season = {(int(row['season']), str(row.get('player_id') or '')) for row in history_rows}

for season in EXPECTED_COLLEGE_SEASONS:
    audit = season_metadata[season]
    methods = Counter(audit.get('positive_match_methods') or {})
    method_totals.update(methods)
    conflicts = list(audit.get('identity_conflicts') or [])
    unmatched = list(audit.get('unmatched_expected_fbs_picks') or [])
    duplicates = list(audit.get('duplicate_roster_outcome_links') or [])
    omissions = list(audit.get('reviewed_roster_source_omissions') or [])
    aliases = list(audit.get('resolved_cross_id_roster_aliases') or [])
    dropped_count = int(audit.get('cross_id_duplicate_alias_rows_dropped') or 0)
    negative_dropped_count = int(audit.get('negative_cross_id_alias_rows_dropped') or 0)
    negative_group_count = int(audit.get('negative_alias_groups_collapsed') or 0)
    negative_alias_candidate_groups += int(audit.get('negative_alias_candidate_groups') or 0)
    negative_alias_unique_stat_groups += int(audit.get('negative_alias_groups_with_unique_checkpoint_stat_id') or 0)
    negative_alias_multiple_stat_groups += int(audit.get('negative_alias_groups_with_multiple_checkpoint_stat_ids') or 0)
    negative_alias_weak_identity_pairs += int(audit.get('negative_alias_pairs_rejected_weak_identity') or 0)
    within_id_dropped_count = int(audit.get('duplicate_roster_rows_dropped') or 0)
    outcome_conflicts = list(audit.get('cross_id_duplicate_alias_outcome_conflicts') or [])
    identity_conflicts.extend(conflicts)
    unmatched_expected.extend(unmatched)
    duplicate_outcomes.extend(duplicates)
    reviewed_omissions.extend(omissions)
    resolved_aliases.extend((season, alias) for alias in aliases)
    alias_rows_dropped += dropped_count
    negative_alias_rows_dropped += negative_dropped_count
    negative_alias_groups_collapsed += negative_group_count
    within_id_duplicate_rows_dropped += within_id_dropped_count
    negative_alias_dropped_keys.extend((season, str(player_id)) for player_id in audit.get('negative_cross_id_alias_player_ids_dropped', []))
    negative_alias_selected_keys.extend((season, str(player_id)) for player_id in audit.get('negative_cross_id_alias_selected_player_ids', []))
    alias_conflicts.extend(outcome_conflicts)
    linkage_rows.append({
        'season': season,
        'methods': ', '.join(f'{key}:{value}' for key, value in sorted(methods.items())),
        'conflicts': len(conflicts),
        'unmatched': len(unmatched),
        'omissions': len(omissions),
        'positive aliases': dropped_count,
        'negative aliases': negative_dropped_count,
        'negative groups': negative_group_count,
        'within-ID dupes': within_id_dropped_count,
    })
print_table(linkage_rows)
print('\nPositive linkage methods:', dict(sorted(method_totals.items())))

require(not identity_conflicts, f'Identity conflicts remain: {identity_conflicts[:3]}')
require(not unmatched_expected, f'Unexpected FBS picks remain unmatched: {unmatched_expected[:3]}')
require(not duplicate_outcomes, f'Duplicate roster outcome links remain: {duplicate_outcomes[:3]}')
require(not alias_conflicts, f'Alias collapse conflicts with a separate outcome: {alias_conflicts[:3]}')
require(len(reviewed_omissions) == EXPECTED_REVIEWED_ROSTER_OMISSIONS, f'Expected {EXPECTED_REVIEWED_ROSTER_OMISSIONS} reviewed {COLLEGE_PROVIDER} omissions, found {len(reviewed_omissions)}.')
require(alias_rows_dropped == int(metadata.get('drops', {}).get('cross_id_duplicate_alias_rows_dropped') or 0), 'Alias-drop total does not reconcile to metadata drops.')
require(alias_rows_dropped == EXPECTED_POSITIVE_ALIAS_ROWS, f'Expected {EXPECTED_POSITIVE_ALIAS_ROWS} positive alias rows for {COLLEGE_PROVIDER}, found {alias_rows_dropped}.')
require(negative_alias_rows_dropped == int(metadata.get('drops', {}).get('negative_cross_id_alias_rows_dropped') or 0), 'Negative alias-drop total does not reconcile to metadata drops.')
require(negative_alias_rows_dropped == EXPECTED_NEGATIVE_ALIAS_ROWS, f'Expected {EXPECTED_NEGATIVE_ALIAS_ROWS} strong negative alias rows for {COLLEGE_PROVIDER}, found {negative_alias_rows_dropped}.')
require(within_id_duplicate_rows_dropped == int(metadata.get('drops', {}).get('duplicate_roster_rows_dropped') or 0), 'Within-ID duplicate total does not reconcile to metadata drops.')
require(within_id_duplicate_rows_dropped == EXPECTED_WITHIN_ID_DUPLICATE_ROWS, f'Expected {EXPECTED_WITHIN_ID_DUPLICATE_ROWS} within-ID duplicate rows for {COLLEGE_PROVIDER}, found {within_id_duplicate_rows_dropped}.')

collapsed_keys = []
for season, alias in resolved_aliases:
    selected = str(alias.get('selected_player_id') or '')
    dropped_ids = [str(value) for value in alias.get('dropped_player_ids', []) if str(value)]
    if selected:
        require((season, selected) in final_player_season, f'Selected alias row missing from final grain: {season}:{selected}')
    for dropped in dropped_ids:
        require((season, dropped) not in final_player_season, f'Dropped alias still appears in final grain: {season}:{dropped}')
        collapsed_keys.append((season, dropped))
require(len(collapsed_keys) == alias_rows_dropped, 'Collapsed alias identities do not reconcile to alias-drop count.')
require(len(negative_alias_dropped_keys) == negative_alias_rows_dropped, 'Negative alias dropped-ID list does not reconcile to the drop count.')
require(len(set(negative_alias_dropped_keys)) == len(negative_alias_dropped_keys), 'A negative alias dropped identity appears more than once within a season.')
for season_player in negative_alias_dropped_keys:
    require(season_player not in final_player_season, f'Dropped negative alias still appears in final grain: {season_player}')
for season_player in negative_alias_selected_keys:
    require(season_player in final_player_season, f'Selected negative alias identity is missing from final grain: {season_player}')

print(f'\nReviewed {metadata.get("college_source_name", COLLEGE_PROVIDER)} roster-source omissions (excluded from source-defined risk set):')
print_table([{
    'draft': item.get('draft_year'),
    'pick': item.get('draft_ovr'),
    'name': item.get('name'),
    'college': item.get('college'),
    'reason': item.get('reason'),
} for item in reviewed_omissions])
print(f'\nPositive cross-ID aliases: {len(resolved_aliases):,} groups / {alias_rows_dropped:,} rows dropped.')
print(f'Strong negative cross-ID aliases: {negative_alias_groups_collapsed:,} groups / {negative_alias_rows_dropped:,} rows dropped.')
print(f'Negative alias review: {negative_alias_candidate_groups:,} candidate groups; {negative_alias_unique_stat_groups:,} unique-stat groups; {negative_alias_multiple_stat_groups:,} multi-stat groups; {negative_alias_weak_identity_pairs:,} weak-identity pairs rejected.')
print(f'Within-ID duplicate roster rows dropped before final grain: {within_id_duplicate_rows_dropped:,}.')

season | methods                                                                                   | conflicts | unmatched | omissions | positive aliases | negative aliases | negative groups | within-ID dupes


-------+-------------------------------------------------------------------------------------------+-----------+-----------+-----------+------------------+------------------+-----------------+----------------
2016   | sportsdataverse_player_id:229                                                             | 0         | 0         | 1         | 0                | 6                | 6               | 6              
2017   | sportsdataverse_player_id:233                                                             | 0         | 0         | 0         | 0                | 2                | 2               | 1              
2018   | sportsdataverse_player_id:236, sportsdataverse_player_id_resolved_incomplete_name_alias:1 | 0         | 0         | 0         | 1                | 4                | 4               | 4              
2019   | sportsdataverse_player_id:244                                                             | 0         | 0         | 1         | 0                | 8      

## Active-roster benchmark join audit

The benchmark loader retains the full ten-draft Combine risk set for its separate conditional probability, but flags only current-roster drafted survivors with a recorded Combine/pro-day measurement for displayed physical averages and elite/top-64 comparisons. A benchmark-left nonmatch is usually a legitimate non-survivor, so technical join checks focus on key uniqueness, ambiguity, conflicts, cohort invariants, and reconciliation to the active drafted registry.

In [8]:
roster_snapshot_week = benchmark_metadata.get('active_roster_snapshot_week')
roster_snapshot = [
    row for row in roster_raw
    if int(parse_number(row.get('season')) or ACTIVE_ROSTER_SEASON) == ACTIVE_ROSTER_SEASON
    and (roster_snapshot_week is None or parse_number(row.get('week')) == float(roster_snapshot_week))
]
status_counts = Counter(str(row.get('status') or 'UNKNOWN').upper().strip() for row in roster_snapshot)
active_status_rows = sum(status_counts[status] for status in EXPECTED_ACTIVE_STATUSES)

def duplicate_nonempty(rows, field):
    values = [str(row.get(field) or '').strip() for row in rows]
    values = [value for value in values if value]
    return len(values) - len(set(values))

active_registry = [dict(row) for row in benchmark_metadata.get('active_drafted_players', [])]
active_measurement_rows = [row for row in benchmark_history if parse_bool(row.get('benchmark_cohort_eligible')) is True]
elite_active_measurement_rows = [row for row in benchmark_history if parse_bool(row.get('benchmark_elite_cohort_eligible')) is True]
benchmark_keys = [(int(parse_number(row.get('draft_year')) or 0), str(row.get('player_id') or '')) for row in benchmark_history]
active_conflict_rows = [row for row in active_measurement_rows if parse_bool(row.get('active_roster_join_conflict')) is True]
registry_conflicts = [row for row in active_registry if parse_bool(row.get('active_roster_join_conflict')) is True]

join_checks = [
    ('Roster snapshot is nonempty', bool(roster_snapshot), len(roster_snapshot)),
    ('Roster snapshot row count matches metadata', len(roster_snapshot) == int(benchmark_metadata.get('active_roster_rows') or -1), len(roster_snapshot)),
    ('Roster active-status count matches metadata', active_status_rows == int(benchmark_metadata.get('active_roster_status_rows') or -1), active_status_rows),
    ('Roster status distribution matches metadata', dict(status_counts) == dict(benchmark_metadata.get('active_roster_status_counts') or {}), dict(status_counts)),
    ('PFR IDs unique in snapshot', duplicate_nonempty(roster_snapshot, 'pfr_id') == 0, duplicate_nonempty(roster_snapshot, 'pfr_id')),
    ('GSIS IDs unique in snapshot', duplicate_nonempty(roster_snapshot, 'gsis_id') == 0, duplicate_nonempty(roster_snapshot, 'gsis_id')),
    ('Benchmark player-year keys unique', len(benchmark_keys) == len(set(benchmark_keys)), len(benchmark_keys) - len(set(benchmark_keys))),
    ('Active drafted registry count matches metadata', len(active_registry) == int(benchmark_metadata.get('active_drafted_window_rows') or -1), len(active_registry)),
    ('Active measurement cohort count matches metadata', len(active_measurement_rows) == int(benchmark_metadata.get('benchmark_rows') or -1), len(active_measurement_rows)),
    ('Elite active cohort count matches metadata', len(elite_active_measurement_rows) == int(benchmark_metadata.get('benchmark_elite_rows') or -1), len(elite_active_measurement_rows)),
    ('No active benchmark join conflicts', not active_conflict_rows and not registry_conflicts, len(active_conflict_rows) + len(registry_conflicts)),
    ('No full draft-window join conflicts', int(benchmark_metadata.get('draft_window_roster_join_conflicts') or 0) == 0, benchmark_metadata.get('draft_window_roster_join_conflicts')),
    ('No combine-window join conflicts', int(benchmark_metadata.get('combine_window_roster_join_conflicts') or 0) == 0, benchmark_metadata.get('combine_window_roster_join_conflicts')),
    ('Every active benchmark row is drafted', all(parse_bool(row.get('drafted')) is True for row in active_measurement_rows), ''),
    ('Every active benchmark status is included', all(str(row.get('active_roster_status') or '').upper() in EXPECTED_ACTIVE_STATUSES for row in active_measurement_rows), ''),
    ('Every elite-active row is active and top-64', all(parse_bool(row.get('benchmark_cohort_eligible')) is True and (parse_number(row.get('draft_ovr')) or 999) <= 64 for row in elite_active_measurement_rows), ''),
]
print_table([{'check': name, 'status': 'PASS' if passed else 'FAIL', 'detail': detail} for name, passed, detail in join_checks])
for name, passed, detail in join_checks:
    require(passed, f'{name} failed: {detail}')

print('\nRoster index ambiguity (ambiguous keys are rejected, never first-matched):')
print(benchmark_metadata.get('active_roster_ambiguous_index_keys'))
print('Draft-window roster match methods:', benchmark_metadata.get('draft_window_roster_match_counts'))
print('Combine-window roster match methods:', benchmark_metadata.get('combine_window_roster_match_counts'))
print(f'Benchmark measurement coverage among active drafted players: {percentage(len(active_measurement_rows), len(active_registry))}')

check                                            | status | detail                                                  
-------------------------------------------------+--------+---------------------------------------------------------
Roster snapshot is nonempty                      | PASS   | 2930                                                    
Roster snapshot row count matches metadata       | PASS   | 2930                                                    
Roster active-status count matches metadata      | PASS   | 2916                                                    
Roster status distribution matches metadata      | PASS   | {'ACT': 2852, 'RET': 11, 'RES': 36, 'E14': 28, 'CUT': 3}
PFR IDs unique in snapshot                       | PASS   | 0                                                       
GSIS IDs unique in snapshot                      | PASS   | 0                                                       
Benchmark player-year keys unique                | PASS   | 0   

In [9]:
selected_draft_rows = [row for row in draft_raw if int(parse_number(row.get('season')) or 0) in EXPECTED_DRAFT_YEARS and parse_number(row.get('pick')) is not None]
draft_year_set = tuple(sorted({int(parse_number(row.get('season')) or 0) for row in selected_draft_rows}))
require(draft_year_set == EXPECTED_DRAFT_YEARS, f'Draft source does not contain all ten expected classes: {draft_year_set}')
registry_by_year = Counter(int(parse_number(row.get('draft_year')) or 0) for row in active_registry)
benchmark_by_year = Counter(int(parse_number(row.get('draft_year')) or 0) for row in active_measurement_rows)
elite_registry_by_year = Counter(int(parse_number(row.get('draft_year')) or 0) for row in active_registry if (parse_number(row.get('draft_pick')) or 999) <= 64)
elite_benchmark_by_year = Counter(int(parse_number(row.get('draft_year')) or 0) for row in elite_active_measurement_rows)
draft_total_by_year = Counter(int(parse_number(row.get('season')) or 0) for row in selected_draft_rows)

survivorship_rows = []
for year in EXPECTED_DRAFT_YEARS:
    survivorship_rows.append({
        'draft': year,
        'all picks': draft_total_by_year[year],
        'active drafted': registry_by_year[year],
        'active share': percentage(registry_by_year[year], draft_total_by_year[year]),
        'active + measure': benchmark_by_year[year],
        'measure coverage': percentage(benchmark_by_year[year], registry_by_year[year]),
        'active top64': elite_registry_by_year[year],
        'top64 + measure': elite_benchmark_by_year[year],
    })
print_table(survivorship_rows)

rows_2021 = [row for row in benchmark_history if int(parse_number(row.get('draft_year')) or 0) == 2021]
require(rows_2021, 'No 2021 benchmark rows found.')
require(all(row.get('measurement_source') == 'pro_day_nonstandard' for row in rows_2021), 'A 2021 row is not marked as nonstandard pro-day testing.')
require(all(row.get('measurement_source') == 'nfl_combine' for row in benchmark_history if int(parse_number(row.get('draft_year')) or 0) != 2021), 'A non-2021 benchmark row has the wrong testing protocol label.')
print('\n2021 caveat: all 2021 testing rows are labeled pro_day_nonstandard; their decentralized pro-day measurements are not fully comparable to normal in-person Combine years.')

draft | all picks | active drafted | active share | active + measure | measure coverage | active top64 | top64 + measure
------+-----------+----------------+--------------+------------------+------------------+--------------+----------------
2017  | 253       | 47             | 18.6%        | 44               | 93.6%            | 19           | 19             
2018  | 256       | 80             | 31.2%        | 69               | 86.2%            | 33           | 33             
2019  | 254       | 92             | 36.2%        | 81               | 88.0%            | 37           | 36             
2020  | 255       | 120            | 47.1%        | 115              | 95.8%            | 44           | 44             
2021  | 259       | 149            | 57.5%        | 130              | 87.2%            | 52           | 48             
2022  | 262       | 203            | 77.5%        | 180              | 88.7%            | 60           | 60             
2023  | 259       | 216         

In [10]:
COMBINE_METRICS = (
    ('height_in', 'Height in'), ('weight_lb', 'Weight lb'), ('forty_s', '40 s'),
    ('bench_reps', 'Bench'), ('vertical_in', 'Vertical'), ('broad_jump_in', 'Broad'),
    ('three_cone_s', '3-cone'), ('shuttle_s', 'Shuttle'),
)
active_measurement_table = []
for position in POSITION_GROUPS:
    position_active = [row for row in active_measurement_rows if normalize_position(row.get('position')) == position]
    position_elite = [row for row in elite_active_measurement_rows if normalize_position(row.get('position')) == position]
    for metric, label in COMBINE_METRICS:
        values = [parse_number(row.get(metric)) for row in position_active]
        values = [value for value in values if value is not None]
        elite_values = [parse_number(row.get(metric)) for row in position_elite]
        elite_values = [value for value in elite_values if value is not None]
        if not values:
            continue
        active_measurement_table.append({
            'pos': position,
            'metric': label,
            'active n': len(values),
            'active avg': f'{fmean(values):.3f}',
            'top64 n': len(elite_values),
            'top64 avg': f'{fmean(elite_values):.3f}' if elite_values else 'n/a',
        })
print_table(active_measurement_table)

cohort_by_position = Counter(normalize_position(row.get('position')) for row in active_measurement_rows)
elite_by_position = Counter(normalize_position(row.get('position')) for row in elite_active_measurement_rows)
require(dict(cohort_by_position) == dict(benchmark_metadata.get('benchmark_cohort_by_position') or {}), 'Active benchmark position counts do not match metadata.')
require(dict(elite_by_position) == dict(benchmark_metadata.get('benchmark_elite_by_position') or {}), 'Elite-active position counts do not match metadata.')
print(f'\nDisplayed physical averages are active-only: n={len(active_measurement_rows):,}; elite/top-64 active-only n={len(elite_active_measurement_rows):,}. Counts above are metric-specific.')

pos  | metric    | active n | active avg | top64 n | top64 avg
-----+-----------+----------+------------+---------+----------
QB   | Height in | 68       | 74.500     | 36      | 74.639   
QB   | Weight lb | 68       | 219.353    | 36      | 220.556  
QB   | 40 s      | 30       | 4.734      | 16      | 4.690    
QB   | Vertical  | 33       | 32.864     | 16      | 32.594   
QB   | Broad     | 31       | 117.000    | 16      | 118.062  
QB   | 3-cone    | 25       | 7.042      | 12      | 6.977    
QB   | Shuttle   | 27       | 4.329      | 13      | 4.318    
RB   | Height in | 106      | 70.566     | 23      | 70.435   
RB   | Weight lb | 107      | 212.290    | 23      | 214.000  
RB   | 40 s      | 86       | 4.488      | 19      | 4.444    
RB   | Bench     | 42       | 19.381     | 10      | 19.700   
RB   | Vertical  | 84       | 35.655     | 18      | 36.833   
RB   | Broad     | 82       | 122.354    | 16      | 125.375  
RB   | 3-cone    | 23       | 7.020      | 4       | 6.

In [11]:
registry_by_pfr = {}
registry_by_cfb = {}
registry_duplicate_ids = []
for index, player in enumerate(active_registry):
    year = int(parse_number(player.get('draft_year')) or 0)
    for field, target in (('pfr_player_id', registry_by_pfr), ('cfb_player_id', registry_by_cfb)):
        value = str(player.get(field) or '').strip()
        if not value:
            continue
        key = (year, value)
        if key in target and target[key] != index:
            registry_duplicate_ids.append((field, key))
        target[key] = index
require(not registry_duplicate_ids, f'Active registry stable IDs are duplicated: {registry_duplicate_ids[:5]}')

active_college_rows = []
college_join_methods = Counter()
college_join_conflicts = []
matched_registry_indexes = set()
for row in history_rows:
    if not truthy(row.get('drafted')):
        continue
    year = int(row['draft_year'])
    pfr = str(row.get('outcome_nflverse_pfr_player_id') or '').strip()
    cfb = str(row.get('outcome_nflverse_cfb_player_id') or '').strip()
    candidates = []
    if pfr and (year, pfr) in registry_by_pfr:
        candidates.append(('pfr_id', registry_by_pfr[(year, pfr)]))
    if cfb and (year, cfb) in registry_by_cfb:
        candidates.append(('cfb_id', registry_by_cfb[(year, cfb)]))
    indexes = {index for _method, index in candidates}
    if len(indexes) > 1:
        college_join_conflicts.append({'row_id': row.get('row_id'), 'matches': candidates})
        continue
    if len(indexes) == 1:
        method = candidates[0][0]
        registry_index = next(iter(indexes))
        active_college_rows.append(row)
        matched_registry_indexes.add(registry_index)
        college_join_methods[method] += 1
require(not college_join_conflicts, f'Active college-production ID join conflicts remain: {college_join_conflicts[:3]}')
require(len(active_college_rows) == len({row.get('row_id') for row in active_college_rows}), 'Active college-production join expanded the player-season grain.')

active_production_table = []
for position in POSITION_GROUPS:
    position_active = [row for row in active_college_rows if normalize_position(row.get('position')) == position]
    position_elite = [row for row in position_active if (parse_number(row.get('draft_ovr')) or 999) <= 64]
    position_metrics = [feature for feature in college_candidate_features(position) if feature.startswith('prod_')]
    for metric in position_metrics:
        values = [parse_number(row.get(metric)) for row in position_active]
        values = [value for value in values if value is not None]
        elite_values = [parse_number(row.get(metric)) for row in position_elite]
        elite_values = [value for value in elite_values if value is not None]
        if not values:
            continue
        active_production_table.append({
            'pos': position,
            'metric': metric.removeprefix('prod_'),
            'active n': len(values),
            'active avg': f'{fmean(values):.3f}',
            'top64 n': len(elite_values),
            'top64 avg': f'{fmean(elite_values):.3f}' if elite_values else 'n/a',
        })
print_table(active_production_table)
print(f'\nActive drafted registry players: {len(active_registry):,}; ID-linked drafted FBS college rows: {len(active_college_rows):,}; registry players represented: {len(matched_registry_indexes):,}.')
print('Stable-ID college join methods:', dict(sorted(college_join_methods.items())))
print(f'The difference is not treated as a join failure: the active registry includes non-FBS players and the {len(reviewed_omissions)} reviewed {COLLEGE_PROVIDER} roster omissions; metric counts further require checkpoint production.')

pos  | metric              | active n | active avg | top64 n | top64 avg


-----+---------------------+----------+------------+---------+----------
QB   | pass_attempts       | 67       | 344.731    | 35      | 347.086  
QB   | pass_yards          | 67       | 2900.493   | 35      | 2974.343 
QB   | pass_tds            | 67       | 23.373     | 35      | 25.171   
QB   | rush_yards          | 67       | 272.940    | 35      | 337.057  
QB   | completion_pct      | 67       | 0.645      | 35      | 0.648    
QB   | yards_per_attempt   | 67       | 8.420      | 35      | 8.569    
QB   | td_rate             | 67       | 0.067      | 35      | 0.072    
RB   | carries             | 106      | 157.519    | 23      | 190.348  
RB   | rush_yards          | 106      | 894.783    | 23      | 1114.478 
RB   | scrimmage_yards     | 106      | 1072.717   | 23      | 1356.522 
RB   | touchdowns          | 106      | 10.057     | 23      | 13.348   
RB   | yards_per_carry     | 106      | 5.589      | 23      | 5.727    
RB   | yards_per_touch     | 106      | 6.161     

## Fixed three-year NFL contributor outcome audit

The contributor label is retrospective NFL performance: at least 500 offense+defense snaps or 150 special-teams snaps across exactly the draft season and next two regular seasons. It is known only with all three complete partitions and a stable PFR ID. Horizons beyond the latest completed NFL season are right-censored, never negative. These fields may select descriptive active-contributor comparisons but remain outside college draft-probability features.

In [12]:
contributor_metadata = dict(benchmark_metadata.get('nfl_three_year_contributor_outcome') or {})
contributor_status = str(contributor_metadata.get('status') or 'unavailable').lower()
contributor_source_status = str(contributor_metadata.get('source_status') or 'unavailable').lower()
contributor_audit_available = contributor_status in {'pass', 'partial'} and 'known_outcomes' in contributor_metadata
contributor_audit_summary = {}
contributor_year_rows = []
if not contributor_audit_available:
    print('UNAVAILABLE: fixed three-year outcomes have not been materialized from complete nflverse snap partitions.')
    if contributor_metadata.get('error'):
        print('Source detail:', contributor_metadata.get('error'))
else:
    completed_nfl_season = int(contributor_metadata.get('completed_nfl_season'))
    require(int(contributor_metadata.get('horizon_seasons') or 0) == 3, 'NFL contributor horizon is not three seasons.')
    require(int(contributor_metadata.get('primary_snap_threshold') or 0) == 500, 'Primary-unit contributor threshold changed.')
    require(int(contributor_metadata.get('special_teams_snap_threshold') or 0) == 150, 'Special-teams contributor threshold changed.')
    known_contributor_rows = [row for row in benchmark_history if parse_bool(row.get('nfl_three_year_outcome_known')) is True]
    right_censored_contributor_rows = [row for row in benchmark_history if parse_bool(row.get('nfl_three_year_right_censored')) is True]
    mature_unknown_rows = [row for row in benchmark_history if parse_bool(row.get('nfl_three_year_outcome_known')) is not True and parse_bool(row.get('nfl_three_year_right_censored')) is not True]
    require(all(int(parse_number(row.get('nfl_three_year_horizon_end')) or 0) - int(parse_number(row.get('nfl_three_year_horizon_start')) or 0) == 2 for row in benchmark_history), 'A benchmark row has a non-three-season NFL horizon.')
    require(all(int(parse_number(row.get('nfl_three_year_horizon_end')) or 9999) <= completed_nfl_season for row in known_contributor_rows), 'A known label extends beyond the completed NFL season.')
    require(all(int(parse_number(row.get('nfl_three_year_horizon_end')) or 0) > completed_nfl_season for row in right_censored_contributor_rows), 'A censored row already has a complete horizon.')
    require(all(row.get('nfl_three_year_contributor') is None for row in right_censored_contributor_rows), 'A censored row has a contributor label.')
    require(all(parse_bool(row.get('nfl_three_year_contributor')) in {True, False} for row in known_contributor_rows), 'A known outcome lacks a binary label.')
    source_artifacts = list(contributor_metadata.get('source_artifacts') or [])
    if contributor_source_status == 'pass':
        require(tuple(contributor_metadata.get('complete_snap_seasons') or ()) == tuple(range(EXPECTED_DRAFT_YEARS[0], completed_nfl_season + 1)), 'Complete snap partitions are not consecutive.')
        for artifact in source_artifacts:
            season = int(artifact.get('season') or 0)
            path = CACHE_DIR / f'snap_counts_{season}.csv'
            require(path.is_file(), f'Missing declared snap artifact: {path}')
            require(path.stat().st_size == int(artifact.get('bytes') or -1), f'Snap artifact byte mismatch: {path.name}')
            require(file_sha256(path) == artifact.get('sha256'), f'Snap artifact hash mismatch: {path.name}')
            require(artifact.get('complete') is True, f'Snap artifact is not complete: {path.name}')
    for draft_year in EXPECTED_DRAFT_YEARS:
        cohort = [row for row in benchmark_history if int(parse_number(row.get('draft_year')) or 0) == draft_year]
        known = [row for row in cohort if parse_bool(row.get('nfl_three_year_outcome_known')) is True]
        censored = [row for row in cohort if parse_bool(row.get('nfl_three_year_right_censored')) is True]
        contributor_year_rows.append({
            'draft': draft_year, 'rows': len(cohort), 'known': len(known),
            'known coverage': percentage(len(known), len(cohort)),
            'contributors': sum(parse_bool(row.get('nfl_three_year_contributor')) is True for row in known),
            'right-censored': len(censored), 'mature unknown': len(cohort) - len(known) - len(censored),
        })
    active_contributor_registry = list(benchmark_metadata.get('active_contributor_players') or [])
    active_contributor_measurement_rows = [row for row in benchmark_history if parse_bool(row.get('benchmark_active_contributor_cohort_eligible')) is True]
    require(len(active_contributor_registry) == int(benchmark_metadata.get('active_contributor_window_rows') or -1), 'Active contributor registry count differs from metadata.')
    require(len(active_contributor_measurement_rows) == int(benchmark_metadata.get('benchmark_active_contributor_rows') or -1), 'Active measured-contributor count differs from metadata.')
    require(all(parse_bool(row.get('nfl_three_year_outcome_known')) is True and parse_bool(row.get('nfl_three_year_contributor')) is True for row in active_contributor_registry), 'Active contributor registry contains an unknown/noncontributor.')
    contributor_audit_summary = {
        'known': len(known_contributor_rows), 'right_censored': len(right_censored_contributor_rows),
        'mature_unknown': len(mature_unknown_rows), 'metadata_known': int(contributor_metadata.get('known_outcomes') or 0),
        'metadata_contributors': int(contributor_metadata.get('contributors') or 0),
        'active_contributors': len(active_contributor_registry),
        'active_contributors_with_measurements': len(active_contributor_measurement_rows),
    }
    print(f'Source status: {contributor_source_status.upper()}; completed NFL season: {completed_nfl_season}; audited annual artifacts: {len(source_artifacts)}')
    print_table(contributor_year_rows)
    print()
    print(f"Combine-history known-label coverage: {percentage(len(known_contributor_rows), len(benchmark_history))}; right-censored={len(right_censored_contributor_rows):,}; mature unknown={len(mature_unknown_rows):,}.")
    print(f"Union PFR-identity outcomes: known={contributor_audit_summary['metadata_known']:,}; contributors={contributor_audit_summary['metadata_contributors']:,}.")
    print(f"Current-roster contributors: {len(active_contributor_registry):,}; with a combine/pro-day measurement={len(active_contributor_measurement_rows):,}.")

Source status: PASS; completed NFL season: 2025; audited annual artifacts: 9
draft | rows | known | known coverage | contributors | right-censored | mature unknown
------+------+-------+----------------+--------------+----------------+---------------
2017  | 327  | 318   | 97.2%          | 176          | 0              | 9             
2018  | 336  | 295   | 87.8%          | 199          | 0              | 41            
2019  | 336  | 333   | 99.1%          | 175          | 0              | 3             
2020  | 337  | 336   | 99.7%          | 203          | 0              | 1             
2021  | 464  | 329   | 70.9%          | 168          | 0              | 135           
2022  | 324  | 312   | 96.3%          | 188          | 0              | 12            
2023  | 319  | 292   | 91.5%          | 194          | 0              | 27            
2024  | 321  | 0     | 0.0%           | 0            | 321            | 0             
2025  | 329  | 0     | 0.0%           | 0            

## Leakage and population-separation checks

Current roster status is post-draft information. It may select descriptive benchmark rows, but it cannot be copied into the full FBS roster probability artifact or its feature allow-list.

In [13]:
model_features = set(metadata.get('model_feature_fields', []))
outcome_fields = set(metadata.get('outcome_fields', []))
post_checkpoint_tokens = ('forty', 'vertical', 'broad_jump', 'three_cone', 'shuttle', 'bench_reps', 'combine', 'pro_day', 'draft_grade')
current_roster_tokens = ('active_roster', 'benchmark_cohort', 'comparison_cohort', 'current_team', 'roster_status')
post_outcome_tokens = ('nfl_three_year', 'active_contributor')
alignment_errors = sum(int(row['draft_year']) != int(row['season']) + 1 for row in history_rows)
week_zero_cutoff_errors = sum(int(row['feature_cutoff_season']) != int(row['season']) - 1 for row in history_rows)
future_stat_errors = sum(not missing(row.get('stats_source_year')) and int(row['stats_source_year']) > int(row['feature_cutoff_season']) for row in history_rows)
wrong_stage = sum(row.get('model_stage') != metadata.get('model_stage') for row in history_rows)
wrong_week = sum(int(row.get('as_of_week') or -1) != int(metadata.get('as_of_week') or 0) for row in history_rows)
college_current_fields = sorted(field for field in header_fields if any(token in field for token in current_roster_tokens))
feature_current_fields = sorted(field for field in model_features if any(token in field for token in current_roster_tokens))
feature_post_outcome_fields = sorted(field for field in model_features if any(token in field for token in post_outcome_tokens))
contract_audit = metadata.get('contract_audit', {})
leakage_controls = metadata.get('leakage_controls', {})

leakage_checks = [
    ('Outcome fields excluded from model features', not (model_features & outcome_fields), sorted(model_features & outcome_fields)),
    ('Combine/pro-day fields excluded from college model', not any(any(token in field for token in post_checkpoint_tokens) for field in model_features), ''),
    ('Current-roster fields absent from college CSV', not college_current_fields, college_current_fields),
    ('Current-roster fields absent from model features', not feature_current_fields, feature_current_fields),
    ('NFL contributor outcome fields absent from model features', not feature_post_outcome_fields, feature_post_outcome_fields),
    ('Immediate next-draft alignment', alignment_errors == 0, f'{alignment_errors:,} bad rows'),
    ('Week-0 prior-season cutoff', week_zero_cutoff_errors == 0, f'{week_zero_cutoff_errors:,} bad rows'),
    ('Statistics do not exceed cutoff', future_stat_errors == 0, f'{future_stat_errors:,} bad rows'),
    ('Single college model stage', wrong_stage == 0, f'{wrong_stage:,} bad rows'),
    ('Single configured checkpoint week', wrong_week == 0, f'{wrong_week:,} bad rows'),
    ('No positive-only nflverse age feature', leakage_controls.get('nflverse_positive_only_age_used_as_feature') is False, leakage_controls.get('nflverse_positive_only_age_used_as_feature')),
    ('Saved contract audit', contract_audit.get('status') == 'pass', contract_audit.get('failures', [])),
    ('Full roster risk set remains unconditional probability', metadata.get('probability_kind') == 'unconditional_next_draft', metadata.get('probability_kind')),
    ('Combine risk set remains separately conditional', benchmark_metadata.get('probability_kind') == 'conditional_on_combine_invitation', benchmark_metadata.get('probability_kind')),
]
print_table([{'check': name, 'status': 'PASS' if passed else 'FAIL', 'detail': detail} for name, passed, detail in leakage_checks])
for name, passed, detail in leakage_checks:
    require(passed, f'{name} failed: {detail}')

check                                                     | status | detail                           
----------------------------------------------------------+--------+----------------------------------
Outcome fields excluded from model features               | PASS   | []                               
Combine/pro-day fields excluded from college model        | PASS   |                                  
Current-roster fields absent from college CSV             | PASS   | []                               
Current-roster fields absent from model features          | PASS   | []                               
NFL contributor outcome fields absent from model features | PASS   | []                               
Immediate next-draft alignment                            | PASS   | 0 bad rows                       
Week-0 prior-season cutoff                                | PASS   | 0 bad rows                       
Statistics do not exceed cutoff                           | PASS   | 0 ba

## Expanding-window model validation

Every position model uses only strictly earlier draft years for training, feature selection, preprocessing, fitting, and calibration. With a two-year warmup, published per-year holdouts cover 2019–2026. OT/IOL share one source cohort. K, P, and LS retain separate evidence and minimum-positive gates. College numeric gaps use training-fold median imputation with no implicit missingness indicators; `has_recorded_stats` is the sole availability feature.

In [14]:
validation_results = []
validation_details = []
missingness_warnings_by_position = {}
model_instances = {}
require(tuple(COLLEGE_AVAILABILITY_FEATURES) == ('has_recorded_stats',), f'College availability allow-list changed: {COLLEGE_AVAILABILITY_FEATURES}')
if RUN_MODEL_VALIDATION:
    positions = tuple(VALIDATION_POSITIONS or POSITION_GROUPS)
    model_rows = normalized_rows
    for position in positions:
        raw_pool = ('OT', 'IOL') if position in {'OT', 'IOL'} else (position,)
        raw_rows = [row for row in model_rows if row.get('position') in raw_pool]
        raw_positives = sum(parse_bool(row.get('drafted')) is True for row in raw_rows)
        try:
            model = CollegeDraftProbabilityModel(model_rows, position=position, population=metadata.get('row_population', 'FBS roster checkpoint'))
            model_instances[position] = model
            result = model.validation
            require(tuple(result.years) == EXPECTED_DRAFT_YEARS, f'{position} validation did not use all ten draft years: {result.years}')
            require(model.raw_model.use_missing_indicators is False, f'{position} college model enabled implicit missing indicators.')
            require(not model.raw_model.missing_features, f'{position} college model learned implicit missing indicators: {model.raw_model.missing_features}')
            availability_features = {feature for feature in model.features if feature.startswith('has_') or 'missing' in feature or 'availability' in feature}
            require(availability_features <= {'has_recorded_stats'}, f'{position} availability-feature contract changed: {availability_features}')
            require('implicit missing-value indicators are disabled' in result.missingness_policy.lower(), f'{position} missingness policy is not explicit.')
            missingness_warnings_by_position[position] = tuple(result.missingness_audit_warnings)
            status = 'PASS' if result.passes_quality_gate else 'WITHHELD'
            validation_results.append({
                'pos': position,
                'source pool': '+'.join(model.source_position_pool),
                'status': status,
                'rows': f'{result.rows:,}',
                'drafted': result.positives,
                'features': len(model.features),
                'implicit flags': len(model.raw_model.missing_features),
                'ROC AUC': f'{result.roc_auc:.3f}' if result.roc_auc is not None else 'n/a',
                'avg precision': f'{result.average_precision:.3f}' if result.average_precision is not None else 'n/a',
                'Brier': f'{result.brier:.4f}' if result.brier is not None else 'n/a',
                'baseline': f'{result.baseline_brier:.4f}',
            })
            if status != 'PASS':
                validation_details.append((position, result.quality_message))
        except ModelError as exc:
            validation_results.append({
                'pos': position, 'source pool': '+'.join(raw_pool), 'status': 'UNAVAILABLE',
                'rows': f'{len(raw_rows):,}', 'drafted': raw_positives, 'features': '-', 'implicit flags': '-',
                'ROC AUC': '-', 'avg precision': '-', 'Brier': '-', 'baseline': '-',
            })
            validation_details.append((position, f'ModelError: {exc}'))
    print_table(validation_results)
    if validation_details:
        print('\nWithheld/unavailable details:')
        for position, detail in validation_details:
            print(f'- {position}: {detail}')
else:
    print('Model validation skipped because RUN_MODEL_VALIDATION=False.')

core_positions = set(POSITION_GROUPS) - {'K', 'P', 'LS'}
failed_core = [row['pos'] for row in validation_results if row['pos'] in core_positions and row['status'] != 'PASS']
require(not failed_core, f'Core position models failed publish gates: {failed_core}')
require('OT' in model_instances and 'IOL' in model_instances, 'OT/IOL pooled models were not fitted.')
require(tuple(model_instances['OT'].source_position_pool) == ('OT', 'IOL'), 'OT did not use the shared OT/IOL source pool.')
require(tuple(model_instances['IOL'].source_position_pool) == ('OT', 'IOL'), 'IOL did not use the shared OT/IOL source pool.')
require(model_instances['OT'].validation.rows == model_instances['IOL'].validation.rows, 'OT/IOL pooled validation denominators differ.')

all_missingness_warnings = [warning for warnings in missingness_warnings_by_position.values() for warning in warnings]
if COLLEGE_PROVIDER == 'cfbd':
    require(any('weight_lb' in warning and '2017' in warning for warning in all_missingness_warnings), 'Expected CFBD 2017 weight missingness warning was not emitted.')
    require(any('bmi' in warning and '2017' in warning for warning in all_missingness_warnings), 'Expected CFBD 2017 BMI missingness warning was not emitted.')
    require(any('class_year_numeric' in warning and '2017' in warning and '2018' in warning for warning in all_missingness_warnings), 'Expected CFBD 2017–2018 class missingness warning was not emitted.')
print('\nCollege-stage missingness policy')
print(next(iter(model_instances.values())).validation.missingness_policy)
print('Source-era missingness audit warnings (implicit indicators remain disabled):')
if not all_missingness_warnings:
    print(f'- none exceeded the audit threshold for {COLLEGE_PROVIDER}')
for position in POSITION_GROUPS:
    for warning in missingness_warnings_by_position.get(position, ()): print(f'- {position}: {warning}')

print('\nSpecialist support audit')
print_table([row for row in validation_results if row['pos'] in {'K', 'P', 'LS'}])

pos  | source pool | status   | rows   | drafted | features | implicit flags | ROC AUC | avg precision | Brier  | baseline
-----+-------------+----------+--------+---------+----------+----------------+---------+---------------+--------+---------
QB   | QB          | PASS     | 6,035  | 85      | 15       | 0              | 0.968   | 0.495         | 0.0122 | 0.0139  
RB   | RB          | PASS     | 8,955  | 151     | 14       | 0              | 0.959   | 0.372         | 0.0145 | 0.0166  
WR   | WR          | PASS     | 16,813 | 253     | 12       | 0              | 0.963   | 0.343         | 0.0134 | 0.0148  
TE   | TE          | PASS     | 7,429  | 117     | 12       | 0              | 0.952   | 0.298         | 0.0134 | 0.0155  
OT   | OT+IOL      | PASS     | 20,188 | 336     | 8        | 0              | 0.838   | 0.088         | 0.0159 | 0.0164  
IOL  | OT+IOL      | PASS     | 20,188 | 336     | 8        | 0              | 0.838   | 0.088         | 0.0159 | 0.0164  
EDGE | EDGE     

### Per-year past-only validation audit

This cell reads fitted models' validation records. Every holdout's training years must equal all available years strictly before its test year; any current/future training year is a hard failure. The compact table aggregates model-year evaluations.

In [15]:
forward_fold_records = []
future_training_violations = []
forward_unavailable_positions = []
for position in POSITION_GROUPS:
    model = model_instances.get(position)
    validation = getattr(model, 'validation', None) if model is not None else None
    per_year = tuple(getattr(validation, 'per_year', ()) or ()) if validation is not None else ()
    scheme = str(getattr(validation, 'validation_scheme', '') or '') if validation is not None else ''
    if not per_year or not scheme.startswith('expanding-window past-only'):
        forward_unavailable_positions.append(position)
        continue
    declared_years = tuple(int(year) for year in validation.years)
    require(tuple(int(year) for year in validation.evaluated_years) == tuple(int(item['year']) for item in per_year), f'{position} evaluated years differ from per-year records.')
    for fold in per_year:
        test_year = int(fold['year'])
        train_years = tuple(int(year) for year in fold.get('train_years') or ())
        expected_train_years = tuple(year for year in declared_years if year < test_year)
        if train_years != expected_train_years or any(year >= test_year for year in train_years):
            future_training_violations.append({'position': position, 'test_year': test_year, 'train_years': train_years, 'expected': expected_train_years})
        forward_fold_records.append({'position': position, **dict(fold)})
forward_validation_available = bool(forward_fold_records)
forward_year_summary = []
if not forward_validation_available:
    print('UNAVAILABLE: fitted/saved models do not expose expanding-window per-year validation records yet.')
else:
    require(not future_training_violations, f'Future-year training detected: {future_training_violations[:5]}')
    for test_year in sorted({int(row['year']) for row in forward_fold_records}):
        folds = [row for row in forward_fold_records if int(row['year']) == test_year]
        evaluated_rows = sum(int(row['rows']) for row in folds)
        train_spans = sorted({f"{min(row['train_years'])}-{max(row['train_years'])}" for row in folds if row.get('train_years')})
        brier_sum = sum(float(row['brier']) * int(row['rows']) for row in folds if row.get('brier') is not None)
        baseline_sum = sum(float(row['baseline_brier']) * int(row['rows']) for row in folds if row.get('baseline_brier') is not None)
        forward_year_summary.append({
            'test year': test_year, 'models': len(folds), 'training span': ','.join(train_spans),
            'holdout rows': f'{evaluated_rows:,}', 'positives': sum(int(row['positives']) for row in folds),
            'weighted Brier': f'{brier_sum / evaluated_rows:.4f}' if evaluated_rows else 'n/a',
            'weighted baseline': f'{baseline_sum / evaluated_rows:.4f}' if evaluated_rows else 'n/a', 'past-only': 'PASS',
        })
    print_table(forward_year_summary)
    print()
    print(f"No-future-training assertion: PASS across {len(forward_fold_records):,} model-year folds; unavailable positions: {', '.join(forward_unavailable_positions) or 'none'}.")

test year | models | training span | holdout rows | positives | weighted Brier | weighted baseline | past-only
----------+--------+---------------+--------------+-----------+----------------+-------------------+----------
2019      | 14     | 2017-2018     | 17,298       | 271       | 0.0143         | 0.0154            | PASS     
2020      | 14     | 2017-2019     | 17,517       | 288       | 0.0152         | 0.0162            | PASS     
2021      | 14     | 2017-2020     | 17,332       | 283       | 0.0151         | 0.0160            | PASS     
2022      | 14     | 2017-2021     | 18,314       | 277       | 0.0142         | 0.0149            | PASS     
2023      | 14     | 2017-2022     | 18,202       | 281       | 0.0139         | 0.0152            | PASS     
2024      | 14     | 2017-2023     | 18,492       | 289       | 0.0145         | 0.0154            | PASS     
2025      | 14     | 2017-2024     | 18,870       | 284       | 0.0137         | 0.0148            | PASS     
2

## Optional model registry and forecast-ledger audit

When content-addressed releases, promotion decisions, or forecast runs exist under the configured output directory, this cell invokes the project's read-only integrity audit. Before the first tracked update, it reports `UNAVAILABLE` and creates nothing.

In [16]:
tracking_store = TrackingStore(TRACKING_OUTPUT_DIR)
tracking_release_paths = sorted(tracking_store.releases_dir.glob('model_*.json'))
tracking_decision_paths = sorted(tracking_store.decisions_dir.glob('decision_*.json'))
tracking_run_paths = sorted(path for path in tracking_store.ledger_dir.glob('*/week_*/forecast_*') if path.is_dir())
tracking_artifacts_present = bool(tracking_release_paths or tracking_decision_paths or tracking_run_paths or tracking_store.champions_path.is_file())
tracking_status = None
if not tracking_artifacts_present:
    print(f'UNAVAILABLE: no model registry or forecast-ledger artifacts exist under {TRACKING_OUTPUT_DIR.relative_to(REPO_ROOT)} yet.')
else:
    tracking_status = tracking_store.status()
    require(tracking_status.get('ok') is True, f"Tracking integrity audit failed: {tracking_status.get('findings')}")
    counts = dict(tracking_status.get('counts') or {})
    print_table([
        {'artifact': 'model releases', 'count': counts.get('model_releases', 0)},
        {'artifact': 'promotion decisions', 'count': counts.get('promotion_decisions', 0)},
        {'artifact': 'forecast runs', 'count': counts.get('forecast_runs', 0)},
        {'artifact': 'champions', 'count': tracking_status.get('champion_count', 0)},
    ])
    print('Integrity audit: PASS')
    print('Validation health:', tracking_status.get('validation_health'))
    print('Validation schemes:', tracking_status.get('validation_schemes'))
    print('Latest forecast checkpoint:', tracking_status.get('latest_forecast_checkpoint'))

artifact            | count
--------------------+------
model releases      | 100  
promotion decisions | 111  
forecast runs       | 7    
champions           | 30   
Integrity audit: PASS
Validation health: {'passed': 100, 'withheld': 0, 'unknown': 0}
Validation schemes: {'expanding_window_past_only': 100}
Latest forecast checkpoint: {'season': 2026, 'week': 0}


## Takeaways

The final cell summarizes only executed checks. Active averages, contributor comparisons, and elite/top-64 values are current-roster survivor comparisons, not unbiased estimates of NFL success, and are never probability features. Metric-specific sample counts must accompany displayed means. Optional enrichment/tracking layers remain explicitly unavailable until their dated artifacts exist.

In [17]:
failed_leakage_checks = [name for name, passed, _detail in leakage_checks if not passed]
passed_positions = [row['pos'] for row in validation_results if row['status'] == 'PASS']
limited_positions = [row['pos'] for row in validation_results if row['status'] != 'PASS']
print(f'- College risk set: {len(history_rows):,} unique player-seasons across 2016–2025; {drafted_rows:,} drafted outcomes; strict gate PASS.')
print(f'- Outcome reconciliation: {matched_picks:,}/{expected_picks:,} expected FBS picks linked; {len(reviewed_omissions)} exact reviewed {COLLEGE_PROVIDER} roster omissions excluded.')
print(f'- Identity grain: {alias_rows_dropped} positive cross-ID alias rows, {negative_alias_rows_dropped} strong negative cross-ID alias rows, and {within_id_duplicate_rows_dropped} within-ID duplicate rows collapsed.')
print(f'- Artifact/contract integrity: {actual_history_bytes:,} bytes and SHA-256 match the sidecar; probability kind/condition are populated and audited on every row.')
print(f'- Active benchmark: {len(active_measurement_rows):,}/{len(active_registry):,} active drafted players have a Combine/pro-day measurement; elite active top-64 n={len(elite_active_measurement_rows):,}.')
print(f'- Join integrity: zero active/full-window conflicts; stable-ID college production joins={len(active_college_rows):,}; methods={dict(sorted(college_join_methods.items()))}.')
print(f'- Leakage/population separation: {"PASS" if not failed_leakage_checks else "FAIL: " + ", ".join(failed_leakage_checks)}; current-roster fields are excluded from probability inputs.')
if contributor_audit_available:
    print(f"- NFL three-year contributor audit: known combine rows={contributor_audit_summary['known']:,}; right-censored={contributor_audit_summary['right_censored']:,}; mature unknown={contributor_audit_summary['mature_unknown']:,}; active contributor registry={contributor_audit_summary['active_contributors']:,}.")
else:
    print('- NFL three-year contributor audit: UNAVAILABLE pending complete snap-count artifacts.')
if success_feature_audit_available:
    print(f'- Checkpoint success features: shared-transformer/rate parity PASS across {len(SUCCESS_FEATURES)} fields; coverage is reported above.')
else:
    print(f'- Checkpoint success features: UNAVAILABLE for {COLLEGE_PROVIDER} ({success_observed_values} populated values across {len(SUCCESS_FEATURES)} fields).')
if forward_validation_available:
    print(f'- Expanding-window validation: no-future-training PASS across {len(forward_fold_records):,} model-year folds covering {len(forward_year_summary)} held-out draft years.')
else:
    print('- Expanding-window validation: UNAVAILABLE in fitted/saved model metadata.')
if tracking_artifacts_present:
    print(f"- Tracking registry/ledger: {'PASS' if tracking_status and tracking_status.get('ok') else 'FAIL'}; counts={tracking_status.get('counts') if tracking_status else {}}.")
else:
    print('- Tracking registry/ledger: UNAVAILABLE before the first tracked release or forecast run.')
if RUN_MODEL_VALIDATION:
    print(f'- Position models passing publish gates ({len(passed_positions)}): {", ".join(passed_positions) or "none"}.')
    print(f'- Position models withheld/unavailable ({len(limited_positions)}): {", ".join(limited_positions) or "none"}.')
print(f'- Missingness: implicit college-stage indicators are disabled; numeric gaps use training-fold medians; has_recorded_stats is the sole availability feature; {len(all_missingness_warnings)} provider-era coverage warning(s) exceeded the audit threshold.')
print('- 2021 is labeled nonstandard pro-day testing. Active and elite-active means remain survivorship- and recency-biased, favor recent classes and players with recorded tests; use them only as descriptive current-roster prototypes.')

- College risk set: 154,279 unique player-seasons across 2016–2025; 2,404 drafted outcomes; strict gate PASS.
- Outcome reconciliation: 2,404/2,404 expected FBS picks linked; 12 exact reviewed sportsdataverse roster omissions excluded.
- Identity grain: 1 positive cross-ID alias rows, 45 strong negative cross-ID alias rows, and 140 within-ID duplicate rows collapsed.
- Artifact/contract integrity: 74,661,611 bytes and SHA-256 match the sidecar; probability kind/condition are populated and audited on every row.
- Active benchmark: 1,472/1,632 active drafted players have a Combine/pro-day measurement; elite active top-64 n=491.
- Join integrity: zero active/full-window conflicts; stable-ID college production joins=1,527; methods={'pfr_id': 1527}.
- Leakage/population separation: PASS; current-roster fields are excluded from probability inputs.
- NFL three-year contributor audit: known combine rows=2,215; right-censored=969; mature unknown=228; active contributor registry=810.
- Checkpoin